In [ ]:
from pathlib import Path

project_root = Path(r"C:\Users\mi178\OneDrive\바탕 화면\SkillBridge")
env_path = project_root / ".env"

if not env_path.is_file():
    raise FileNotFoundError(f".env 파일을 찾지 못했어요. 확인할 위치: {env_path}")

api_key = ""
for line in env_path.read_text(encoding="utf-8-sig").splitlines():
    name, separator, value = line.partition("=")
    if separator and name.strip() == "WORK24_API_KEY":
        api_key = value.strip()
        break

if not api_key:
    raise ValueError(".env에 WORK24_API_KEY=발급받은_인증키 형식으로 입력해 주세요.")

print("인증키를 읽었어요.")

인증키를 읽었어요.


In [10]:
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import urlopen
from xml.etree import ElementTree

api_base = "https://www.work24.go.kr/cm/openApi/call/wk"


def fetch_xml(endpoint, params):
    query = urlencode({"authKey": api_key, "returnType": "XML", **params})

    try:
        with urlopen(f"{endpoint}?{query}", timeout=15) as response:
            raw_xml = response.read()
    except HTTPError as error:
        raise RuntimeError(f"고용24 요청 실패 (HTTP {error.code})") from None
    except (URLError, TimeoutError):
        raise RuntimeError("고용24 서버에 연결하지 못했어요.") from None

    try:
        return raw_xml, ElementTree.fromstring(raw_xml)
    except ElementTree.ParseError:
        raise RuntimeError("고용24 응답을 XML로 읽지 못했어요.") from None


max_pages = 5
page_size = 10
keywords = ("개발", "소프트웨어", "프로그래머", "백엔드", "프론트엔드", "sw", "developer")

posts = []

for page in range(1, max_pages + 1):
    _, list_root = fetch_xml(
        f"{api_base}/callOpenApiSvcInfo210L21.do",
        {"callTp": "L", "startPage": page, "display": page_size},
    )

    if list_root.tag != "dhsOpenEmpInfoList":
        raise RuntimeError(f"{page}페이지 조회 실패. 공채속보 신청 상태를 확인해 주세요.")

    page_posts = list_root.findall("dhsOpenEmpInfo")
    posts.extend(page_posts)

    if len(page_posts) < page_size:
        break

candidates = [
    post for post in posts
    if any(keyword in post.findtext("empWantedTitle", "").lower()
           for keyword in keywords)
]

print(f"조회한 공고 {len(posts)}건 · 제목으로 찾은 후보 {len(candidates)}건")

for post in candidates:
    number = post.findtext("empSeqno", "")
    company = post.findtext("empBusiNm", "")
    title = post.findtext("empWantedTitle", "")
    print(f"{number} | {company} | {title}")

if not candidates:
    print("제목에서 후보를 찾지 못했어요.")

조회한 공고 50건 · 제목으로 찾은 후보 2건
185557 | 에스케이시그넷 | 전기차 충전기 SW/FW 검증 및 시험 자동화 엔지니어
185517 | 삼성이엔에이 | 경력사원 채용(화공 사업관리, 연구개발, 분석)


In [11]:
selected_id = "185557"

selected = next(
    (post for post in posts if post.findtext("empSeqno") == selected_id),
    None,
)

if selected is None:
    raise ValueError("현재 목록에 출력된 공고 번호를 입력해 주세요.")

print("공고 번호", selected_id)
print("회사명", selected.findtext("empBusiNm", ""))
print("제목", selected.findtext("empWantedTitle", ""))
print("공고 링크", selected.findtext("empWantedHomepgDetail", ""))

공고 번호 185557
회사명 에스케이시그넷
제목 전기차 충전기 SW/FW 검증 및 시험 자동화 엔지니어
공고 링크 https://www.skcareers.com/Recruit/Detail/R262043


In [12]:
from urllib.parse import parse_qsl, urlsplit

detail_example_url = "https://www.work24.go.kr/cm/openApi/call/wk/callOpenApiSvcInfo210D21.do?authKey=[인증키]&returnType=XML&callTp=D&empSeqno=[공개채용공고순번]"

example = urlsplit(detail_example_url.strip())
detail_params = dict(parse_qsl(example.query))
detail_params.pop("authKey", None)
detail_params.pop("returnType", None)

id_fields = [
    name for name, value in detail_params.items()
    if value.startswith("[") and value.endswith("]")
]

if (
    example.scheme != "https"
    or example.hostname != "www.work24.go.kr"
    or detail_params.get("callTp") != "D"
    or len(id_fields) != 1
):
    raise ValueError("공채속보상세 탭의 기본검색 예제 URL을 그대로 넣어 주세요.")

detail_params[id_fields[0]] = selected_id
detail_endpoint = example._replace(query="", fragment="").geturl()
detail_bytes, detail_root = fetch_xml(detail_endpoint, detail_params)

# 제목이 상세 응답에 들어 있는지 확인한 뒤 원본을 저장한다.
if not (detail_root.findtext(".//empWantedTitle") or "").strip():
    print("응답 태그", detail_root.tag)
    print("하위 태그", [child.tag for child in detail_root])
    raise RuntimeError("상세 응답에서 공고 제목을 확인하지 못했어요.")

raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
sample_path = raw_dir / f"work24_sample_{selected_id}.xml"
sample_path.write_bytes(detail_bytes)

print("저장 위치", sample_path.relative_to(project_root))

저장 위치 data\raw\work24_sample_185557.xml
